# Colab Session: bench-r22
Generated from colab-cli history log.

**Session Created**: 2026-11-09 09:29:01
- Endpoint: `gpu-a100-s-usc1c0-2x3doximutcx9`
- Hardware: `A100`

*File Operation*: `upload` on `/content/PREREG_R1.sha256`

*File Operation*: `upload` on `/content/PREREG_R2.sha256`

*File Operation*: `upload` on `/content/PREREG_R22.sha256`

*File Operation*: `upload` on `/content/METHOD_AND_THEORY_R22.md`

*File Operation*: `upload` on `/content/smoke_r22.py`

*File Operation*: `upload` on `/content/SMOKE_R22_RESULTS.json`

*File Operation*: `upload` on `/content/colab_bench_r2.py`

*File Operation*: `upload` on `/content/colab_bench_r22.py`

In [ ]:
from pathlib import Path
Path('/content/research_sources/qwen35_08b_inst').mkdir(parents=True, exist_ok=True)


*File Operation*: `upload` on `/content/research_sources/qwen35_08b_inst/tokenizer.json`

*File Operation*: `upload` on `/content/r2_mech_dev.tar.gz`

In [ ]:
import tarfile
tarfile.open('/content/r2_mech_dev.tar.gz').extractall('/content')


In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_REDACTED_TOKEN'
os.environ['USE_HUB_KERNELS'] = 'OFF'
import os, shutil, subprocess, sys
os.environ["USE_HUB_KERNELS"] = "OFF"
print("[vm-setup-r22] Installing locked transformers commit 0f7a8768733c448751f25abcb4b6b3ffef829cf5...", flush=True)
uv_bin = shutil.which("uv")
pkgs = [
    "https://github.com/huggingface/transformers/archive/0f7a8768733c448751f25abcb4b6b3ffef829cf5.tar.gz",
    "accelerate",
    "safetensors",
]
cmd = [uv_bin, "pip", "install", "--system", *pkgs] if uv_bin else [sys.executable, "-m", "pip", "install", "-q", *pkgs]
subprocess.run(cmd, check=True)

import torch
assert torch.cuda.is_available(), "CUDA unavailable on bench-r22"
props = torch.cuda.get_device_properties(0)
uuid_str = str(getattr(props, "uuid", "")).strip()
print(f"[vm-setup-r22] GPU={props.name}, VRAM={props.total_memory / (2**30):.2f} GiB, UUID={uuid_str}", flush=True)
assert "A100" in props.name and props.total_memory >= 38 * (2**30), f"Expected A100 >=38GiB, got {props.name}"
assert uuid_str, "Empty CUDA UUID"

subprocess.run([sys.executable, "-u", "/content/colab_bench_r22.py", "--verify-prereg", "/content/PREREG_R22.sha256", "--cpu-self-test"], check=True)
print("[vm-setup-r22] A100 hardware, PyTorch tiny-arch self-test, smoke_r22, and PREREG_R22.sha256 verified.", flush=True)


In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, "-u", "/content/colab_bench_r22.py", "--verify-prereg", "/content/PREREG_R22.sha256", "--cpu-self-test"], capture_output=True, text=True)
print("RC:", res.returncode)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)


In [ ]:
from pathlib import Path
import subprocess, sys

src = Path("/content/colab_bench_r22.py").read_text()
old_block = """    delta32_arch = {
        "mode": "Delta32",
        "C": C,
        "chunks": full_arch["chunks"],
        "base_cache": full_arch["base_cache"],
        "anchor_conv": full_arch["anchor_conv"],
        "anchor_S": full_arch["anchor_S"],
        "end_S": full_arch["end_S"],
        "P": {l: {c: full_arch["PH"][l][c][0] for c in full_arch["PH"][l]} for l in engine.gdn},
    }
    delta16_arch = {
        "mode": "Delta16",
        "C": C,
        "chunks": full_arch["chunks"],
        "base_cache": full_arch["base_cache"],
        "anchor_conv": full_arch["anchor_conv"],
        "anchor_S": {c: {l: full_arch["anchor_S"][c][l].to(dtype=torch.bfloat16) for l in engine.gdn} for c in full_arch["anchor_S"]},
        "end_S": {c: {l: full_arch["end_S"][c][l].to(dtype=torch.bfloat16) for l in engine.gdn} for c in full_arch["end_S"]},
        "P": {l: {c: full_arch["PH"][l][c][0].to(dtype=torch.bfloat16) for c in full_arch["PH"][l]} for l in engine.gdn},
    }"""

new_block = """    delta32_arch = {
        "archive_mode": "Delta32",
        "C": C,
        "chunks": full_arch["chunks"],
        "base_cache": full_arch["base_cache"],
        "anchor_conv": full_arch["anchor_conv"],
        "anchor_S": full_arch["anchor_S"],
        "end_conv": full_arch["end_conv"],
        "P": {c: {l: full_arch["PH"][c][l][0] for l in engine.gdn} for c in full_arch["PH"]},
    }
    delta16_arch = {
        "archive_mode": "Delta16",
        "C": C,
        "chunks": full_arch["chunks"],
        "base_cache": full_arch["base_cache"],
        "anchor_conv": full_arch["anchor_conv"],
        "anchor_S": {c: {l: full_arch["anchor_S"][c][l].to(dtype=torch.bfloat16) for l in engine.gdn} for c in full_arch["anchor_S"]},
        "end_conv": full_arch["end_conv"],
        "P": {c: {l: full_arch["PH"][c][l][0].to(dtype=torch.bfloat16) for l in engine.gdn} for c in full_arch["PH"]},
    }"""

assert old_block in src, "old_block not found"
Path("/tmp/test_colab_bench_r22.py").write_text(src.replace(old_block, new_block, 1))
res = subprocess.run([sys.executable, "-u", "/tmp/test_colab_bench_r22.py", "--cpu-self-test"], cwd="/content", capture_output=True, text=True)
print("RC:", res.returncode)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)


In [ ]:
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH="/content")
res = subprocess.run([sys.executable, "-u", "/tmp/test_colab_bench_r22.py", "--cpu-self-test"], cwd="/content", env=env, capture_output=True, text=True)
print("RC:", res.returncode)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)


In [ ]:
import sys, tempfile
sys.path.insert(0, "/tmp")
sys.path.insert(0, "/content")
import test_colab_bench_r22 as m
with tempfile.TemporaryDirectory() as tmp_d:
    res = m.run_gpu_benchmark_r22(tmp_d, ["0.8B"], device="cpu", tiny_arch_dryrun=True)
    m0 = res["models"]["0.8B"]
    assert m0["Module1_G2_5Way_Isolation"]["64"]["pass_conv_proj_matched_exact_zero"] is True
    assert m0["Module1_G2_5Way_Isolation"]["64"]["max_S_F_vs_seg_chunked"] < 1e-5
    arch_m = m0["Module2_Runtime_Archives_And_Scaling"]["T128_C64"]["archive_modes"]
    assert arch_m["Delta16"]["extra_archive_mib"] < arch_m["Delta32"]["extra_archive_mib"] < arch_m["Direct32"]["extra_archive_mib"]
    print("[self-test-r22] Tiny-arch 4-module execution verified PASS!")


In [ ]:
from pathlib import Path
Path("/tmp/test_colab_bench_r22.py").unlink(missing_ok=True)


*File Operation*: `upload` on `/content/PREREG_R22.sha256`

*File Operation*: `upload` on `/content/colab_bench_r22.py`

In [ ]:
import os
os.environ['USE_HUB_KERNELS'] = 'OFF'
import subprocess, sys
subprocess.run([sys.executable, "-u", "/content/colab_bench_r22.py", "--verify-prereg", "/content/PREREG_R22.sha256", "--cpu-self-test"], check=True)


In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_REDACTED_TOKEN'
os.environ['USE_HUB_KERNELS'] = 'OFF'
os.environ['RUN_MODEL'] = '0.8B'
import json, os, subprocess, sys
from pathlib import Path

tag = os.environ["RUN_MODEL"]
summary_p = Path(f"/content/bench_r22_out/model_{tag}/summary_model_r22.json")
if summary_p.is_file():
    sm = json.loads(summary_p.read_text())
    if sm.get("status") == "COMPLETE":
        print(f"[remote-launch-r22] Model {tag} ALREADY COMPLETE on VM; skipping re-run.", flush=True)
        Path(f"/content/run_r22_{tag}.rc").write_text("0\n")
        raise SystemExit(0)

pid_file = Path(f"/content/run_r22_{tag}.pid")
rc_file = Path(f"/content/run_r22_{tag}.rc")
log_file = Path(f"/content/run_r22_{tag}.log")

if pid_file.is_file() and not rc_file.is_file():
    old_pid = int(pid_file.read_text().strip())
    stat_p = Path(f"/proc/{old_pid}/stat")
    if stat_p.is_file() and " Z " not in stat_p.read_text():
        print(f"[remote-launch-r22] Model {tag} already actively running at PID {old_pid}.", flush=True)
        raise SystemExit(0)

rc_file.unlink(missing_ok=True)
log_file.write_text("")
env = dict(os.environ)
env["USE_HUB_KERNELS"] = "OFF"
runner_code = f"""
import subprocess, sys
from pathlib import Path
log_f = open({str(log_file)!r}, "a", buffering=1)
cmd = [
    sys.executable, "-u", "/content/colab_bench_r22.py",
    "--verify-prereg", "/content/PREREG_R22.sha256",
    "--out-dir", "/content/bench_r22_out",
    "--models", {tag!r},
]
rc = subprocess.call(cmd, stdout=log_f, stderr=subprocess.STDOUT)
log_f.flush()
log_f.close()
Path({str(rc_file)!r}).write_text(f"{{rc}}\\n")
"""
proc = subprocess.Popen(
    [sys.executable, "-u", "-c", runner_code],
    env=env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
pid_file.write_text(f"{proc.pid}\n")
print(f"[remote-launch-r22] Started detached GPU worker for {tag} (PID={proc.pid})", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '0'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '658'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '784'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '784'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1315'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1315'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1315'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1419'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1471'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1471'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1471'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1471'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1523'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1523'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1523'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1523'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1523'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1523'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1523'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1585'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1585'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1585'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1585'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1585'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1585'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1585'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1647'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1647'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1647'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1700'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1700'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1700'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1753'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1753'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1753'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1807'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '0.8B'
os.environ['LOG_OFFSET'] = '1807'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['TARGET_MODEL'] = '0.8B'
import os, tarfile
from pathlib import Path

tag = os.environ["TARGET_MODEL"]
out = Path("/content/bench_r22_out")
tar_path = Path(f"/content/pull_r22_{tag}.tar.gz")
with tarfile.open(tar_path, "w:gz") as tf:
    mdir = out / f"model_{tag}"
    if mdir.exists():
        tf.add(mdir, arcname=f"bench_r22_out/model_{tag}")
    for top in ("session_hardware_lock.json", "summary_r22.json"):
        p = out / top
        if p.exists():
            tf.add(p, arcname=f"bench_r22_out/{top}")
print(f"[pull-r22] Created {tar_path} ({tar_path.stat().st_size / 1024:.1f} KiB)", flush=True)


*File Operation*: `download` on `/content/pull_r22_0.8B.tar.gz`

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_REDACTED_TOKEN'
os.environ['USE_HUB_KERNELS'] = 'OFF'
os.environ['RUN_MODEL'] = '2B'
import json, os, subprocess, sys
from pathlib import Path

tag = os.environ["RUN_MODEL"]
summary_p = Path(f"/content/bench_r22_out/model_{tag}/summary_model_r22.json")
if summary_p.is_file():
    sm = json.loads(summary_p.read_text())
    if sm.get("status") == "COMPLETE":
        print(f"[remote-launch-r22] Model {tag} ALREADY COMPLETE on VM; skipping re-run.", flush=True)
        Path(f"/content/run_r22_{tag}.rc").write_text("0\n")
        raise SystemExit(0)

pid_file = Path(f"/content/run_r22_{tag}.pid")
rc_file = Path(f"/content/run_r22_{tag}.rc")
log_file = Path(f"/content/run_r22_{tag}.log")

if pid_file.is_file() and not rc_file.is_file():
    old_pid = int(pid_file.read_text().strip())
    stat_p = Path(f"/proc/{old_pid}/stat")
    if stat_p.is_file() and " Z " not in stat_p.read_text():
        print(f"[remote-launch-r22] Model {tag} already actively running at PID {old_pid}.", flush=True)
        raise SystemExit(0)

rc_file.unlink(missing_ok=True)
log_file.write_text("")
env = dict(os.environ)
env["USE_HUB_KERNELS"] = "OFF"
runner_code = f"""
import subprocess, sys
from pathlib import Path
log_f = open({str(log_file)!r}, "a", buffering=1)
cmd = [
    sys.executable, "-u", "/content/colab_bench_r22.py",
    "--verify-prereg", "/content/PREREG_R22.sha256",
    "--out-dir", "/content/bench_r22_out",
    "--models", {tag!r},
]
rc = subprocess.call(cmd, stdout=log_f, stderr=subprocess.STDOUT)
log_f.flush()
log_f.close()
Path({str(rc_file)!r}).write_text(f"{{rc}}\\n")
"""
proc = subprocess.Popen(
    [sys.executable, "-u", "-c", runner_code],
    env=env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
pid_file.write_text(f"{proc.pid}\n")
print(f"[remote-launch-r22] Started detached GPU worker for {tag} (PID={proc.pid})", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '0'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '801'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '861'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1389'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1389'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1389'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1439'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1439'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1439'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1489'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1489'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1539'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1539'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1539'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1539'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1589'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1589'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1589'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1589'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1589'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1589'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1589'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1649'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1649'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1649'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1649'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1649'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1649'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1649'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1709'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1709'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1709'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1760'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1760'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1760'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1811'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1811'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1811'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1863'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1863'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '2B'
os.environ['LOG_OFFSET'] = '1863'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['TARGET_MODEL'] = '2B'
import os, tarfile
from pathlib import Path

tag = os.environ["TARGET_MODEL"]
out = Path("/content/bench_r22_out")
tar_path = Path(f"/content/pull_r22_{tag}.tar.gz")
with tarfile.open(tar_path, "w:gz") as tf:
    mdir = out / f"model_{tag}"
    if mdir.exists():
        tf.add(mdir, arcname=f"bench_r22_out/model_{tag}")
    for top in ("session_hardware_lock.json", "summary_r22.json"):
        p = out / top
        if p.exists():
            tf.add(p, arcname=f"bench_r22_out/{top}")
print(f"[pull-r22] Created {tar_path} ({tar_path.stat().st_size / 1024:.1f} KiB)", flush=True)


*File Operation*: `download` on `/content/pull_r22_2B.tar.gz`

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_REDACTED_TOKEN'
os.environ['USE_HUB_KERNELS'] = 'OFF'
os.environ['RUN_MODEL'] = '4B'
import json, os, subprocess, sys
from pathlib import Path

tag = os.environ["RUN_MODEL"]
summary_p = Path(f"/content/bench_r22_out/model_{tag}/summary_model_r22.json")
if summary_p.is_file():
    sm = json.loads(summary_p.read_text())
    if sm.get("status") == "COMPLETE":
        print(f"[remote-launch-r22] Model {tag} ALREADY COMPLETE on VM; skipping re-run.", flush=True)
        Path(f"/content/run_r22_{tag}.rc").write_text("0\n")
        raise SystemExit(0)

pid_file = Path(f"/content/run_r22_{tag}.pid")
rc_file = Path(f"/content/run_r22_{tag}.rc")
log_file = Path(f"/content/run_r22_{tag}.log")

if pid_file.is_file() and not rc_file.is_file():
    old_pid = int(pid_file.read_text().strip())
    stat_p = Path(f"/proc/{old_pid}/stat")
    if stat_p.is_file() and " Z " not in stat_p.read_text():
        print(f"[remote-launch-r22] Model {tag} already actively running at PID {old_pid}.", flush=True)
        raise SystemExit(0)

rc_file.unlink(missing_ok=True)
log_file.write_text("")
env = dict(os.environ)
env["USE_HUB_KERNELS"] = "OFF"
runner_code = f"""
import subprocess, sys
from pathlib import Path
log_f = open({str(log_file)!r}, "a", buffering=1)
cmd = [
    sys.executable, "-u", "/content/colab_bench_r22.py",
    "--verify-prereg", "/content/PREREG_R22.sha256",
    "--out-dir", "/content/bench_r22_out",
    "--models", {tag!r},
]
rc = subprocess.call(cmd, stdout=log_f, stderr=subprocess.STDOUT)
log_f.flush()
log_f.close()
Path({str(rc_file)!r}).write_text(f"{{rc}}\\n")
"""
proc = subprocess.Popen(
    [sys.executable, "-u", "-c", runner_code],
    env=env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
pid_file.write_text(f"{proc.pid}\n")
print(f"[remote-launch-r22] Started detached GPU worker for {tag} (PID={proc.pid})", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '0'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '113'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '502'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1016'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1076'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1076'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1076'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1605'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1605'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1605'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1605'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1655'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1655'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1655'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1705'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1705'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1705'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1705'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1755'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1755'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1755'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1755'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1755'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1805'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1865'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1925'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1925'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1925'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1925'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1976'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1976'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1976'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '1976'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2027'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2027'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2027'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2027'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2027'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2080'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2080'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2080'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '4B'
os.environ['LOG_OFFSET'] = '2080'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['TARGET_MODEL'] = '4B'
import os, tarfile
from pathlib import Path

tag = os.environ["TARGET_MODEL"]
out = Path("/content/bench_r22_out")
tar_path = Path(f"/content/pull_r22_{tag}.tar.gz")
with tarfile.open(tar_path, "w:gz") as tf:
    mdir = out / f"model_{tag}"
    if mdir.exists():
        tf.add(mdir, arcname=f"bench_r22_out/model_{tag}")
    for top in ("session_hardware_lock.json", "summary_r22.json"):
        p = out / top
        if p.exists():
            tf.add(p, arcname=f"bench_r22_out/{top}")
print(f"[pull-r22] Created {tar_path} ({tar_path.stat().st_size / 1024:.1f} KiB)", flush=True)


*File Operation*: `download` on `/content/pull_r22_4B.tar.gz`

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_REDACTED_TOKEN'
os.environ['USE_HUB_KERNELS'] = 'OFF'
os.environ['RUN_MODEL'] = '9B'
import json, os, subprocess, sys
from pathlib import Path

tag = os.environ["RUN_MODEL"]
summary_p = Path(f"/content/bench_r22_out/model_{tag}/summary_model_r22.json")
if summary_p.is_file():
    sm = json.loads(summary_p.read_text())
    if sm.get("status") == "COMPLETE":
        print(f"[remote-launch-r22] Model {tag} ALREADY COMPLETE on VM; skipping re-run.", flush=True)
        Path(f"/content/run_r22_{tag}.rc").write_text("0\n")
        raise SystemExit(0)

pid_file = Path(f"/content/run_r22_{tag}.pid")
rc_file = Path(f"/content/run_r22_{tag}.rc")
log_file = Path(f"/content/run_r22_{tag}.log")

if pid_file.is_file() and not rc_file.is_file():
    old_pid = int(pid_file.read_text().strip())
    stat_p = Path(f"/proc/{old_pid}/stat")
    if stat_p.is_file() and " Z " not in stat_p.read_text():
        print(f"[remote-launch-r22] Model {tag} already actively running at PID {old_pid}.", flush=True)
        raise SystemExit(0)

rc_file.unlink(missing_ok=True)
log_file.write_text("")
env = dict(os.environ)
env["USE_HUB_KERNELS"] = "OFF"
runner_code = f"""
import subprocess, sys
from pathlib import Path
log_f = open({str(log_file)!r}, "a", buffering=1)
cmd = [
    sys.executable, "-u", "/content/colab_bench_r22.py",
    "--verify-prereg", "/content/PREREG_R22.sha256",
    "--out-dir", "/content/bench_r22_out",
    "--models", {tag!r},
]
rc = subprocess.call(cmd, stdout=log_f, stderr=subprocess.STDOUT)
log_f.flush()
log_f.close()
Path({str(rc_file)!r}).write_text(f"{{rc}}\\n")
"""
proc = subprocess.Popen(
    [sys.executable, "-u", "-c", runner_code],
    env=env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
pid_file.write_text(f"{proc.pid}\n")
print(f"[remote-launch-r22] Started detached GPU worker for {tag} (PID={proc.pid})", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '0'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '57'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '113'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '113'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '113'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '952'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '952'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1014'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1014'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1014'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1014'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1077'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1608'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1608'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1608'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1608'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1658'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1658'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1658'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1708'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1708'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1708'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1708'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1943'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1943'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1943'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1943'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1943'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1943'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '1993'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2053'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2114'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2114'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2114'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2114'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2166'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2166'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2166'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2166'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2218'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2218'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2218'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2218'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2218'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2271'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2271'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2271'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2271'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['RUN_MODEL'] = '9B'
os.environ['LOG_OFFSET'] = '2271'
import os
from pathlib import Path

tag = os.environ["RUN_MODEL"]
offset = int(os.environ["LOG_OFFSET"])
log_file = Path(f"/content/run_r22_{tag}.log")
rc_file = Path(f"/content/run_r22_{tag}.rc")
pid_file = Path(f"/content/run_r22_{tag}.pid")

data = log_file.read_bytes() if log_file.is_file() else b""
new_bytes = data[offset:]
new_offset = len(data)
if new_bytes:
    print(new_bytes.decode("utf-8", errors="replace"), end="", flush=True)

if rc_file.is_file():
    rc_val = int(rc_file.read_text().strip())
    print(f"\n__COLAB_POLL_STATUS__:DONE:{rc_val}:{new_offset}", flush=True)
else:
    alive = False
    if pid_file.is_file():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
            alive = True
        except OSError:
            alive = False
    if alive:
        print(f"\n__COLAB_POLL_STATUS__:RUNNING:0:{new_offset}", flush=True)
    else:
        print(f"\n__COLAB_POLL_STATUS__:DEAD:1:{new_offset}", flush=True)


In [ ]:
import os
os.environ['TARGET_MODEL'] = '9B'
import os, tarfile
from pathlib import Path

tag = os.environ["TARGET_MODEL"]
out = Path("/content/bench_r22_out")
tar_path = Path(f"/content/pull_r22_{tag}.tar.gz")
with tarfile.open(tar_path, "w:gz") as tf:
    mdir = out / f"model_{tag}"
    if mdir.exists():
        tf.add(mdir, arcname=f"bench_r22_out/model_{tag}")
    for top in ("session_hardware_lock.json", "summary_r22.json"):
        p = out / top
        if p.exists():
            tf.add(p, arcname=f"bench_r22_out/{top}")
print(f"[pull-r22] Created {tar_path} ({tar_path.stat().st_size / 1024:.1f} KiB)", flush=True)


*File Operation*: `download` on `/content/pull_r22_9B.tar.gz`

In [ ]:
import subprocess, sys, tarfile
from pathlib import Path

subprocess.run([
    sys.executable, "-u", "/content/colab_bench_r22.py",
    "--verify-prereg", "/content/PREREG_R22.sha256",
    "--aggregate-only",
    "--out-dir", "/content/bench_r22_out",
], check=True)

tar_path = Path("/content/bench_r22_results_all.tar.gz")
with tarfile.open(tar_path, "w:gz") as tf:
    tf.add("/content/bench_r22_out", arcname="bench_r22_out")
print(f"[finalize-r22] Packed {tar_path} ({tar_path.stat().st_size / (1024*1024):.2f} MiB)", flush=True)


*File Operation*: `download` on `/content/bench_r22_results_all.tar.gz`